# 24_Packet_P007_Diversified_Panel_Lock
## Materia Arche Agent OS v3.0

**Work Packet ID:** P-007
**Decision this packet informs:** Can we lock the diversified 3-candidate panel for Phase 2 outreach?
**Fastest falsifier:** Verify that all 3 diversified candidates (850, 1064, 119) pass the same P-005 robustness criteria and export a final locked shortlist.
**Success threshold:** All 3 candidates have ≥80% top-20 test appearance rate, T80 ≥500h, and cover ≥2 composition families
**Failure / kill threshold:** Any candidate fails robustness re-check
**Reuse requirement if it fails:** Document which candidates failed and return to P-006 pool
**Dataset ID/version:** perovskite_stability_clean.csv (1,543 devices)
**Benchmark ID:** P-006 diversified panel recommendation
**Owner:** ML Scientist
**Reviewer:** Evidence Guardian
**Release ceiling:** Internal

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
from collections import Counter
import yaml
import warnings
warnings.filterwarnings('ignore')

with open('contracts/schema.lock.yaml') as f:
    schema = yaml.safe_load(f)
with open('contracts/model.lock.yaml') as f:
    model_lock = yaml.safe_load(f)

print("Libraries loaded — Packet P-007")

Libraries loaded — Packet P-007


## 1. Load data and define diversified panel

In [2]:
df = pd.read_csv('perovskite_stability_clean.csv')
X = df[schema['columns']].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

# Diversified panel from P-006
panel = {
    850:  {'family': 'MA-only',       'rationale': 'P-005 top pick — most rank-stable device'},
    1064: {'family': 'MA-FA mixed',   'rationale': 'Diversity pick — FA₀.₈₅MA₀.₁₅, highest T80 in pool'},
    119:  {'family': 'FA-Cs (triple)', 'rationale': 'Diversity pick — FA₀.₈₃Cs₀.₁₇, triple-cation stability'},
}

print(f"Dataset: {len(df)} devices")
print(f"Diversified panel: {list(panel.keys())}")
print(f"Families: {[v['family'] for v in panel.values()]}")

Dataset: 1543 devices
Diversified panel: [850, 1064, 119]
Families: ['MA-only', 'MA-FA mixed', 'FA-Cs (triple)']


## 2. Verify robustness (test-set-only ranking, 20 splits)

In [3]:
et_params = model_lock['params'].copy()
n_splits = 20
seeds = list(range(1, n_splits + 1))

device_test_appearances = Counter()
device_top20_counts = Counter()
device_test_ranks = {idx: [] for idx in panel.keys()}

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    et = ExtraTreesRegressor(random_state=42, **et_params)
    et.fit(X_tr, y_tr)
    pred_test = et.predict(X_te)
    test_ranks = pd.Series(pred_test, index=X_te.index).rank(ascending=False).astype(int)
    
    for idx in panel.keys():
        if idx in X_te.index:
            device_test_appearances[idx] += 1
            rank = test_ranks[idx]
            device_test_ranks[idx].append(rank)
            if rank <= 20:
                device_top20_counts[idx] += 1

print("=" * 80)
print("ROBUSTNESS VERIFICATION — DIVERSIFIED PANEL")
print("=" * 80)

all_pass = True
panel_results = []

for idx, info in panel.items():
    n_app = device_test_appearances[idx]
    n_top20 = device_top20_counts[idx]
    rate = n_top20 / n_app if n_app > 0 else 0
    ranks = device_test_ranks[idx]
    t80 = df.iloc[idx]['Stability_PCE_T80']
    
    passed = rate >= 0.80 and t80 >= 500 and n_app >= 3
    if not passed:
        all_pass = False
    
    status = "PASS" if passed else "FAIL"
    print(f"\n  Device {idx} ({info['family']}) — {status}")
    print(f"    Actual T80: {t80:.0f}h")
    print(f"    Top-20 rate: {n_top20}/{n_app} ({rate:.0%})")
    if ranks:
        print(f"    Mean test rank: {np.mean(ranks):.1f} +/- {np.std(ranks):.1f}")
        print(f"    Range: [{np.min(ranks)}, {np.max(ranks)}]")
    print(f"    Rationale: {info['rationale']}")
    
    panel_results.append({
        'device_idx': idx,
        'family': info['family'],
        'actual_T80': t80,
        'test_appearances': n_app,
        'top20_count': n_top20,
        'top20_rate': round(rate, 3),
        'mean_test_rank': round(np.mean(ranks), 1) if ranks else None,
        'std_test_rank': round(np.std(ranks), 1) if ranks else None,
        'rationale': info['rationale'],
        'status': status,
    })

print(f"\nAll pass: {all_pass}")

ROBUSTNESS VERIFICATION — DIVERSIFIED PANEL

  Device 850 (MA-only) — PASS
    Actual T80: 3400h
    Top-20 rate: 4/4 (100%)
    Mean test rank: 1.0 +/- 0.0
    Range: [1, 1]
    Rationale: P-005 top pick — most rank-stable device

  Device 1064 (MA-FA mixed) — PASS
    Actual T80: 5400h
    Top-20 rate: 5/5 (100%)
    Mean test rank: 3.4 +/- 3.0
    Range: [1, 9]
    Rationale: Diversity pick — FA₀.₈₅MA₀.₁₅, highest T80 in pool

  Device 119 (FA-Cs (triple)) — PASS
    Actual T80: 3423h
    Top-20 rate: 4/4 (100%)
    Mean test rank: 9.5 +/- 1.8
    Range: [7, 12]
    Rationale: Diversity pick — FA₀.₈₃Cs₀.₁₇, triple-cation stability

All pass: True


## 3. Add prediction intervals

In [4]:
# Train on frozen split for prediction intervals
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
et = ExtraTreesRegressor(random_state=42, **et_params)
et.fit(X_tr, y_tr)

# Per-tree predictions for panel devices
tree_preds = np.array([tree.predict(X) for tree in et.estimators_])

print("=" * 80)
print("PREDICTION INTERVALS (from 700 ET trees)")
print("=" * 80)

for i, result in enumerate(panel_results):
    idx = result['device_idx']
    preds = tree_preds[:, idx]
    pred_mean = np.expm1(np.mean(preds))
    pred_p10 = np.expm1(np.percentile(preds, 10))
    pred_p90 = np.expm1(np.percentile(preds, 90))
    
    panel_results[i]['pred_mean_hours'] = round(pred_mean, 0)
    panel_results[i]['pred_p10_hours'] = round(pred_p10, 0)
    panel_results[i]['pred_p90_hours'] = round(pred_p90, 0)
    
    print(f"  Device {idx}: {pred_mean:.0f}h (80% CI: {pred_p10:.0f}–{pred_p90:.0f}h)")

PREDICTION INTERVALS (from 700 ET trees)
  Device 850: 3118h (80% CI: 2735–3400h)
  Device 1064: 5027h (80% CI: 5400–6735h)
  Device 119: 1886h (80% CI: 1499–3423h)


## 4. Full composition profiles

In [5]:
comp_cols = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl', 'MA', 'FA', 'Cs']

print("=" * 80)
print("LOCKED DIVERSIFIED PANEL — FULL PROFILES")
print("=" * 80)

for i, result in enumerate(panel_results, 1):
    idx = result['device_idx']
    row = df.iloc[idx]
    
    print(f"\n  Candidate #{i} — Device {idx} [{result['family']}]")
    print(f"    {result['rationale']}")
    print(f"    Actual T80: {result['actual_T80']:.0f}h")
    print(f"    Robustness: top-20 in {result['top20_count']}/{result['test_appearances']} tests ({result['top20_rate']:.0%})")
    print(f"    Mean test rank: {result['mean_test_rank']} +/- {result['std_test_rank']}")
    print(f"    Prediction: {result['pred_mean_hours']:.0f}h (80% CI: {result['pred_p10_hours']:.0f}–{result['pred_p90_hours']:.0f}h)")
    print(f"    Composition:")
    ma = row['MA'] if not np.isnan(row['MA']) else 0
    fa = row['FA'] if not np.isnan(row['FA']) else 0
    cs = row['Cs'] if not np.isnan(row['Cs']) else 0
    pb = row['Pb'] if not np.isnan(row['Pb']) else 0
    sn = row['Sn'] if not np.isnan(row['Sn']) else 0
    i_val = row['I'] if not np.isnan(row['I']) else 0
    br = row['Br'] if not np.isnan(row['Br']) else 0
    cl = row['Cl'] if not np.isnan(row['Cl']) else 0
    
    # Build formula string
    a_parts = []
    if ma > 0: a_parts.append(f'MA{ma:.2f}' if ma != 1 else 'MA')
    if fa > 0: a_parts.append(f'FA{fa:.2f}' if fa != 1 else 'FA')
    if cs > 0: a_parts.append(f'Cs{cs:.2f}' if cs != 1 else 'Cs')
    a_str = ''.join(a_parts) if a_parts else '?'
    
    b_parts = []
    if pb > 0: b_parts.append(f'Pb{pb:.0f}' if pb != 1 else 'Pb')
    if sn > 0: b_parts.append(f'Sn{sn:.0f}' if sn != 1 else 'Sn')
    b_str = ''.join(b_parts) if b_parts else '?'
    
    x_parts = []
    if i_val > 0: x_parts.append(f'I{i_val:.0f}' if i_val != 1 else 'I')
    if br > 0: x_parts.append(f'Br{br:.0f}' if br != 1 else 'Br')
    if cl > 0: x_parts.append(f'Cl{cl:.0f}' if cl != 1 else 'Cl')
    x_str = ''.join(x_parts) if x_parts else '?'
    
    print(f"      Formula: {a_str} | {b_str} | {x_str}")
    print(f"      Bandgap: {row['Perovskite_band_gap']}")
    print(f"      Voc={row['JV_default_Voc']}, Jsc={row['JV_default_Jsc']}, FF={row['JV_default_FF']}")

LOCKED DIVERSIFIED PANEL — FULL PROFILES

  Candidate #1 — Device 850 [MA-only]
    P-005 top pick — most rank-stable device
    Actual T80: 3400h
    Robustness: top-20 in 4/4 tests (100%)
    Mean test rank: 1.0 +/- 0.0
    Prediction: 3118h (80% CI: 2735–3400h)
    Composition:
      Formula: MA3.00 | Pb4 | I13
      Bandgap: nan
      Voc=1.01, Jsc=19.53, FF=0.764

  Candidate #2 — Device 1064 [MA-FA mixed]
    Diversity pick — FA₀.₈₅MA₀.₁₅, highest T80 in pool
    Actual T80: 5400h
    Robustness: top-20 in 5/5 tests (100%)
    Mean test rank: 3.4 +/- 3.0
    Prediction: 5027h (80% CI: 5400–6735h)
    Composition:
      Formula: MA0.15FA0.85 | Pb | I3Br0
      Bandgap: 1.6
      Voc=1.02, Jsc=21.17, FF=0.6509999999999999

  Candidate #3 — Device 119 [FA-Cs (triple)]
    Diversity pick — FA₀.₈₃Cs₀.₁₇, triple-cation stability
    Actual T80: 3423h
    Robustness: top-20 in 4/4 tests (100%)
    Mean test rank: 9.5 +/- 1.8
    Prediction: 1886h (80% CI: 1499–3423h)
    Composition:
  

## 5. Export locked panel

In [6]:
# Export final locked panel
panel_export = pd.DataFrame(panel_results)
panel_export.to_csv('Packet_P007_Locked_Panel.csv', index=False)
print("Packet_P007_Locked_Panel.csv exported")
print("")
print(panel_export[['device_idx', 'family', 'actual_T80', 'top20_rate', 'mean_test_rank', 'pred_mean_hours', 'status']].to_string(index=False))

Packet_P007_Locked_Panel.csv exported

 device_idx         family  actual_T80  top20_rate  mean_test_rank  pred_mean_hours status
        850        MA-only      3400.0         1.0             1.0           3118.0   PASS
       1064    MA-FA mixed      5400.0         1.0             3.4           5027.0   PASS
        119 FA-Cs (triple)      3423.0         1.0             9.5           1886.0   PASS


## 6. Honest status footer

In [7]:
n_families = len(set(r['family'] for r in panel_results))

if all_pass and n_families >= 2:
    status = "Confirmed"
    decision = "Advance"
elif all_pass:
    status = "Promising"
    decision = "Iterate"
else:
    status = "Negative"
    decision = "Stop"

print("=" * 65)
print("HONEST STATUS — MATERIA ARCHE v3.0")
print("=" * 65)
print(f"Packet ID: P-007")
print(f"Status: {status}")
print(f"Evidence level: E4 (validation-ready package)")
print(f"Decision outcome: {decision}")
print(f"Release ceiling: Internal")
print(f"Domain: perovskite")
print(f"Dataset version: perovskite_stability_clean.csv (1,543 devices)")
print(f"Benchmark: P-005 robustness + P-006 diversity")
print(f"This run: {sum(1 for r in panel_results if r['status']=='PASS')}/3 candidates pass, {n_families} composition families")

if all_pass and n_families >= 2:
    print(f"What worked: Diversified panel passes robustness and diversity checks")
    print(f"What failed or remains uncertain: Prediction intervals remain wide; bandgap data missing for some candidates")
    print(f"What changes next: Panel is locked for Phase 2 outreach")
elif all_pass:
    print(f"What worked: All candidates pass robustness")
    print(f"What failed or remains uncertain: Diversity still insufficient")
    print(f"What changes next: Return to P-006 pool for more diversity")
else:
    failed = [r['device_idx'] for r in panel_results if r['status']=='FAIL']
    print(f"What worked: Identified which candidates pass")
    print(f"What failed or remains uncertain: Devices {failed} failed robustness re-check")
    print(f"What changes next: Replace failed candidates from P-006 pool")

print(f"Reusable asset created: Packet_P007_Locked_Panel.csv")
print(f"Safeguard added: Panel locked with robustness + diversity verification")
print(f"")
print(f"Reviewer sign-off: Evidence Guardian __________")

HONEST STATUS — MATERIA ARCHE v3.0
Packet ID: P-007
Status: Confirmed
Evidence level: E4 (validation-ready package)
Decision outcome: Advance
Release ceiling: Internal
Domain: perovskite
Dataset version: perovskite_stability_clean.csv (1,543 devices)
Benchmark: P-005 robustness + P-006 diversity
This run: 3/3 candidates pass, 3 composition families
What worked: Diversified panel passes robustness and diversity checks
What failed or remains uncertain: Prediction intervals remain wide; bandgap data missing for some candidates
What changes next: Panel is locked for Phase 2 outreach
Reusable asset created: Packet_P007_Locked_Panel.csv
Safeguard added: Panel locked with robustness + diversity verification

Reviewer sign-off: Evidence Guardian __________
